In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
from pathlib import Path
import seaborn as sns 
import matplotlib as mpl
from scipy.stats import linregress
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter

In [ ]:
# Import data 
        # This is where you will define filelst and filenames
parent_folder = Path(r".../data_aeris_manuscript/figure_7/data") # UPDATE FOR YOUR FILE PATH
filelst = list(parent_folder.glob("*.csv"))
filenames = [f.name for f in filelst]

# Read each file into a data frame
data_dict = {}
for i in range(len(filelst)):
    data_dict[filenames[i]] = pd.read_csv(filelst[i])


In [8]:
filenames

['100kda_6pt5mm_C001H001S0001_20260203_153329.csv',
 '100kda_6pt5mm_C001H001S0001_20260203_161633.csv',
 '100kda_6pt5mm_C001H001S0001_20260203_162105.csv',
 '100kda_6pt5mm_C001H001S0001_20260203_163725.csv',
 '100kda_gel_C001H001S0001_20260203_191754.csv',
 '100kda_gel_C001H001S0001_20260203_192250.csv',
 '100kda_gel_C001H001S0001_20260203_192720.csv',
 '100kda_Mg_C001H001S0001_20260203_171108.csv',
 '100kda_Mg_C001H001S0001_20260203_171524.csv',
 '100kda_Mg_C001H001S0001_20260203_172002.csv',
 '100kda_Mg_C001H001S0001_20260203_174131.csv',
 '100kda_NaOH_C001H001S0001_20260203_180855.csv',
 '100kda_NaOH_C001H001S0001_20260203_181322.csv',
 '100kda_NaOH_C001H001S0001_20260203_181740.csv',
 '250kda_C001H001S0001_20260203_151235.csv',
 '250kda_C001H001S0001_20260203_164159.csv',
 '250kda_C001H001S0001_20260203_164640.csv',
 '250kda_gel_C001H001S0001_20260203_193729.csv',
 '250kda_gel_C001H001S0001_20260203_194220.csv',
 '250kda_gel_C001H001S0001_20260203_194900.csv',
 '250kda_Mg_C001H001S

In [ ]:
# reading the excel file with all the run information into data frame
# dictionary with list of filenames associated with a given experimental condition 

# Example method of organizing data is provided below: 

mws:list = ["50kda", "100kda", "250kda", "500kda"] ## THIS IS UNIQUE MOLECULAR WEIGHTS
combo:list = ["mg", "naoh", "gel", "ha"]  ## these are the tested conditions

org_expts = {}
for mw in mws:
    mg_lst = []
    naoh_lst = []
    gel_lst = []
    ha_lst = []
    for f in filenames: # for each file
        if mw in f[:len(mw)]:     # check if it is of a particular molecular weight
            # Creating individual lists for each condition in combo
            if "mg" in f.lower():
                mg_lst.append(f)
            elif "naoh" in f.lower():
                naoh_lst.append(f)
            elif "gel" in f.lower():
                gel_lst.append(f)
            else:
                ha_lst.append(f)
        # Assembling all lists into dictionary
        org_expts.update({mw+"_mg": mg_lst, 
                        mw+"_naoh": naoh_lst,
                        mw+"_gel": gel_lst,
                        mw+"_ha": ha_lst})



In [ ]:
## REQUIRED PREPROCESSING BEFORE RUNNING BELOW: 

# for each experiment, compute "tc-t" and "d/d0, mm".
d0 = 6 # mm (plate diameter)

# eg below
for f in filenames[:-1]:
    data_dict[f]["d/d0, mm"] = data_dict[f]["diameter_mm"]/d0  # computing true d/do
    data_dict[f]["tc-t"] = data_dict[f]["time_ms"].iloc[-1] - data_dict[f]["time_ms"]# Computing tbreak - t, which is the time delta between breakage and the current time point.


In [10]:
# Averaging the relevant data, but aligning by tc-t instead, and in ascending order: 

data_cols = ["time_ms", "tc-t", "diameter_mm", "d/d0, mm"]

# Compiling data into a dictionary of dataframes
averaged_tc_aligned = {}
for mw in mws: # for each molecular weight
    # combine Mg
    dfs = []
    for i in range(len(org_expts[mw+"_"+combo[0]])):
        id = mw+"_"+combo[0]
        df = data_dict[org_expts[id][i]].sort_values(by = "tc-t").reset_index(drop = True)
        mask = df["diameter_mm"].notna()
        dfs.append(df[data_cols][mask])
        averaged_tc_aligned.update({id: pd.concat(dfs, axis = 1)})
    # combine NaOH
    dfs = []
    for i in range(len(org_expts[mw+"_"+combo[1]])):
        id = mw+"_"+combo[1]
        df = data_dict[org_expts[id][i]].sort_values(by = "tc-t").reset_index(drop = True)
        mask = df["diameter_mm"].notna()
        dfs.append(df[data_cols][mask])
        averaged_tc_aligned.update({id: pd.concat(dfs, axis = 1)})
    # combine gel
    dfs = []
    for i in range(len(org_expts[mw+"_"+combo[2]])):
        id = mw+"_"+combo[2]
        df = data_dict[org_expts[id][i]].sort_values(by = "tc-t").reset_index(drop = True)
        mask = df["diameter_mm"].notna()
        dfs.append(df[data_cols][mask])
        averaged_tc_aligned.update({id: pd.concat(dfs, axis = 1)})
    # combine HA only 
    dfs = []
    for i in range(len(org_expts[mw+"_"+combo[3]])):
        id = mw+"_"+combo[3]
        df = data_dict[org_expts[id][i]].sort_values(by = "tc-t").reset_index(drop = True)
        mask = df["diameter_mm"].notna()
        dfs.append(df[data_cols][mask])
        averaged_tc_aligned.update({id: pd.concat(dfs, axis = 1)})


# # #averaging - aligned experiment time

# # note, in this dataset, length can only be 3 or 4
for mw in mws:
    for cond in combo:
        if len(org_expts[mw+"_"+cond]) == 3: # three columns to average
        # if the length is 3 then do this
            averaged_tc_aligned[mw+"_"+ cond]["avg_expt_time"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [0,4,8]].mean(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["std_expt_time"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [0,4,8]].std(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["avg_tc-t"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [1,5,9]].mean(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["std_tc-t"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [1,5,9]].std(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["avg_d, mm"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [2,6,10]].mean(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["std_d, mm"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [2,6,10]].std(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["avg_d/d0, mm"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [3,7,11]].mean(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["std_d/d0, mm"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [3,7,11]].std(axis = 1)
        else:   # else four columns to average
            averaged_tc_aligned[mw+"_"+ cond]["avg_expt_time"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [0,4,8,12]].mean(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["std_expt_time"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [0,4,8,12]].std(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["avg_tc-t"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [1,5,9,13]].mean(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["std_tc-t"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [1,5,9,13]].std(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["avg_d, mm"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [2,6,10,14]].mean(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["std_d, mm"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [2,6,10,14]].std(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["avg_d/d0, mm"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [3,7,11,15]].mean(axis = 1)
            averaged_tc_aligned[mw+"_"+ cond]["std_d/d0, mm"] = averaged_tc_aligned[mw+"_"+ cond].iloc[:, [3,7,11,15]].std(axis = 1)

## TEST
test_data = mws[1]+"_"+combo[2]
print("test data frame:", test_data)
averaged_tc_aligned[test_data]

test data frame: 100kda_gel


,time_ms,tc-t,diameter_mm,"d/d0, mm",time_ms,tc-t,diameter_mm,"d/d0, mm",time_ms,tc-t,diameter_mm,"d/d0, mm",avg_expt_time,std_expt_time,avg_tc-t,std_tc-t,"avg_d, mm","std_d, mm","avg_d/d0, mm","std_d/d0, mm"
0,62.0,0.0,0.000,0.000000,43.0,0.0,0.000,0.000000,44.0,0.0,0.000,0.000000,49.666667,10.692677,0.0,0.0,0.000000,0.000000,0.000000,0.000000
1,61.0,1.0,0.215,0.035833,42.0,1.0,0.129,0.021500,43.0,1.0,0.387,0.064500,48.666667,10.692677,1.0,0.0,0.243667,0.131367,0.040611,0.021895
2,60.0,2.0,0.559,0.093167,41.0,2.0,0.559,0.093167,42.0,2.0,0.731,0.121833,47.666667,10.692677,2.0,0.0,0.616333,0.099304,0.102722,0.016551
3,59.0,3.0,0.817,0.136167,40.0,3.0,0.774,0.129000,41.0,3.0,0.903,0.150500,46.666667,10.692677,3.0,0.0,0.831333,0.065684,0.138556,0.010947
4,58.0,4.0,0.989,0.164833,39.0,4.0,0.989,0.164833,40.0,4.0,1.032,0.172000,45.666667,10.692677,4.0,0.0,1.003333,0.024826,0.167222,0.004138
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,4.0,58.0,2.580,0.430000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,NaN,58.0,NaN,2.580000,NaN,0.430000,NaN
59,3.0,59.0,2.623,0.437167,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000,NaN,59.0,NaN,2.623000,NaN,0.437167,NaN
60,2.0,60.0,2.666,0.444333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,NaN,60.0,NaN,2.666000,NaN,0.444333,NaN
61,1.0,61.0,2.666,0.444333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,61.0,NaN,2.666000,NaN,0.444333,NaN


In [12]:
# Goal: calculate relaxation times

# Relevant equation: D(t)/D0 proportional to Ae^(-t/3*relaxation time -> ln(D(t)/D0)) = -t/3*relaxation time
# To calculate, want: slope of log/linear plot of D(t)/D0 vs (tc-t). Slope = 1/3*relaxation time --> relaxation time = 1/3*slope

### MAKE SURE YOUR COLUMN INDEX VALUES ARE ACCURATE BELOW ###

## 1. Calculate ln(D(t)/D0) for each experiment

klist = [] # generating iterable list of keys 
for mw in mws:
    for cond in combo:
        klist.append(mw+"_"+cond)

relax_expts = {} # dictionary to manage the dataframes with relevant data for relaxation time 
for i in range(len(klist)): # Going through each relevant experiment
    df = averaged_tc_aligned[klist[i]]
    if len(org_expts[klist[i]]) == 3:
        # column selection
        t_cols = [1, 5, 9] # columns corresponding to tc-t for each replicate
        d_cols = [3, 7, 11] # columns corresponding to d/do for each replicate
        for c in d_cols[:]: 
            df[f"ln(d/d0)_{d_cols.index(c)+1}"] = np.log(df.iloc[1:, c]) # caclulating lnd/do. each column has new title lnd/do_1 to lnd/do_3
        df["avgln(d/d0)"] = df.iloc[:, [20,21,22]].mean(axis=1) # mean of the ln(d/do)
        df["stdln(d/d0)"] = df.iloc[:, [20,21,22]].std(axis=1)  # standard deviation of ln(d/do)

        relax_df = df.iloc[:, [14,20,21,22,23,24]].dropna() # Only keep data frame 
        relax_expts.update({klist[i]: relax_df})
    else:
        # column selection
        t_cols = [1, 5, 9, 13] # columns corresponding to tc-t for each replicate
        d_cols = [3, 7, 11, 15] # columns corresponding to d/do for each replicate
        for c in d_cols[:]: 
            df[f"ln(d/d0)_{d_cols.index(c)+1}"] = np.log(df.iloc[1:, c]) # caclulating lnd/do. each column has new title lnd/do_1 to lnd/do_3
        df["avgln(d/d0)"] = df.iloc[:, [24,25,26, 27]].mean(axis=1) # mean of the ln(d/do)
        df["stdln(d/d0)"] = df.iloc[:, [24,25,26, 27]].std(axis=1)  # standard deviation of ln(d/do)

        relax_df = df.iloc[:, [18,24,25,26,27,28,29]].dropna() # Only keep data frame 
        relax_expts.update({klist[i]: relax_df})

# # 2. Determine linear region via second derivative analysis, and then computing relaxation times for each  data set, and assembling into a dataframe
rel_data = {} # Dictionary to store regression parameters, with key = Experiment ID, and value = {(start index, end index) 
for i in range(len(klist)): # Going through each relevant experiment
    relax_df = relax_expts[klist[i]]
    reps = len(org_expts[klist[i]])

    #Smoothing window selection for savgol filter
    if len(relax_df) < 50:
        smooth_window = 7
    elif len(relax_df) < 200:
        smooth_window = 15
    else:
        smooth_window = 51
    # print(klist[i],"length = ", len(relax_df), "smooth_window =", smooth_window)
    #smooth and take second derivative
    relax_df["smooth_avglnd/do"] = savgol_filter(relax_df["avgln(d/d0)"], window_length=smooth_window, polyorder=3)  # smoothing using savitzky golay
    relax_df["d1"] = np.gradient(relax_df["smooth_avglnd/do"], relax_df["avg_tc-t"]) # find first derivative
    relax_df["smooth_d1"] = savgol_filter(relax_df["d1"], window_length=smooth_window, polyorder=3)  # smoothing using savitzky golay
    relax_df["d2"] = np.gradient(relax_df["smooth_d1"], relax_df["avg_tc-t"]) # find second derivative
    relax_df["fwd tc-t"] = relax_df["avg_tc-t"].values[-1] - relax_df["avg_tc-t"]

    # Identifying best second derivative threshold value 
    # Initializing 
    thresholds = np.logspace(-5, -2, 50) 
    best_rsq = 0.9 # starting r square  

    for thresh in thresholds:   # Testing different thresholds 
        mask = np.abs(relax_df["d2"].values[:]) < thresh  # selecting for second derivative values lower than threshold
        # Skipping anything that has too small a range
        if mask.sum() < 10:
            continue

        if klist[i] == "250kda_gel": ### EXCEPTION!!
            start = relax_df["avg_tc-t"][mask].index[0] + 300 # Exception
        else:
            start = relax_df["avg_tc-t"][mask].index[0]    # find starting index for linear regression
        if klist[i] == "500kda_gel": ### EXCEPTION!!
            end = relax_df["avg_tc-t"][mask].index[-1] -1000 # Exception
        else:
            end = relax_df["avg_tc-t"][mask].index[-1]    # find final index for linear regression
        rvalues = [] # list of r_values
        # regression for each rep
        for c in range(1,reps+1): 
            slope, intercept, r_value, p_value, stderr = linregress(relax_df.loc[:, "avg_tc-t"].values[start:end], relax_df.iloc[:, c].values[start:end]) # Do linear regression
            rvalues.append(r_value)
        r_value = (sum(rvalues)/reps)
        r_sq = r_value**2

        #If the r_squared value is greater than the previously existing r_squared value, update linear range parameters, and see if there is a better window
        if r_sq > best_rsq:
            best_rsq = r_sq
            start_ind = start
            end_ind = end
            threshhold = thresh
            # print("threshold=", thresh, "rsq = ", r_sq, "current best rsq =", best_rsq, "Better rsquared so trying different windows")
            for t in range(0, int(len(relax_df)*0.2), int((len(relax_df)*0.1)/20)+1): # Testing for an optimal window, with constraint that the window has to be atleast 50% of the dataset to update the regression parameters
                test_start = start + t
                rvalues = [] # list of tuples with slope, intercept, r^2 
                for c in range(1,reps+1):
                    slope, intercept, r_value, p_value, stderr = linregress(relax_df.loc[:, "avg_tc-t"].values[test_start:end], relax_df.iloc[:, c].values[test_start:end]) # Do linear regression
                    rvalues.append(r_value)
                r_value = (sum(rvalues)/reps)
                window_rsq = r_value**2
                windowsize = end - test_start
                # print("rsq = ", window_rsq, "current best rsq = ", best_rsq, "window size=", windowsize)
                if window_rsq >= best_rsq and windowsize >= int(len(relax_df)*0.5) :
                    best_rsq = window_rsq
                    start_ind = test_start
                    end_ind = end
                    threshhold = thresh
                    # print("satisfied both constraints, updating rsq to", best_rsq,"with windowsize = ", end_ind-start_ind)

    # analysis:
    rvalues = [] # list of tuples with slope, intercept, r^2 
    slopes = []
    reltimes = []
    intercepts = []
    for c in range(1,reps+1):
        slope, intercept, r_value, p_value, stderr = linregress(relax_df.loc[:, "avg_tc-t"].values[start_ind:end_ind], relax_df.iloc[:, c].values[start_ind:end_ind]) # Do linear regression
        rvalues.append(r_value)
        slopes.append(slope)
        reltimes.append(1/(3*slope))
        intercepts.append(intercept)
    rel_data.update({klist[i]: {"start_ind": start_ind, 
                                "end_ind": end_ind,
                                "N": reps,
                                "rsq" :sum(rvalues)/reps,
                                "slope": sum(slopes)/reps,
                                "intercept": sum(intercepts)/reps,
                                "avg relaxation time, ms": sum(reltimes)/reps,
                                 "sd relaxation time": np.std(reltimes)}})   

# Get all analyzed relaxation data values into a dataframe
relaxation_t_df = pd.DataFrame(rel_data).T.reset_index().rename(columns = {"index": "experiment_id"})

# Getting average tbreak and final filament diameter
avg_tbreaks = []
sd_tbreaks = []
avg_d_breaks = []
sd_d_breaks = []
avg_d_prebreaks = []
sd_d_prebreaks = []
avg_d_inits = []
sd_d_inits=[]
for i in range(len(klist)):
    key = klist[i]
    reps = len(org_expts[key])
    tbreaks = [] # all breakage times
    d_breaks = [] # filament diameters at break time (this will be used to determine if it broke at the end of the experiment or no) - 0 means broken, >0 means not broken
    d_pre_break = [] # Filament diameters right before breaking
    d_inits = [] # initial filament diameter after stretch complete
    for rep in range(reps):
        tbreaks.append(data_dict[org_expts[key][rep]].loc[:, "time_ms"].values[-1]) # get tbreak, ms and add to list
        d_breaks.append(data_dict[org_expts[key][rep]].loc[:, "diameter_mm"].values[-1]) # get diameter at end of experiment (d_breaks)
        d_pre_break.append(data_dict[org_expts[key][rep]].loc[:, "diameter_mm"].values[-2]) # get diameter right before breakage
        d_inits.append(data_dict[org_expts[key][rep]].loc[:, "diameter_mm"].values[0]) # get diameter right before breakage
    avg_tbreaks.append(np.mean(tbreaks))
    sd_tbreaks.append(np.std(tbreaks))
    avg_d_breaks.append(np.mean(d_breaks))
    sd_d_breaks.append(np.std(d_breaks))    
    avg_d_prebreaks .append(np.mean(d_pre_break))
    sd_d_prebreaks.append(np.std(d_pre_break))    
    avg_d_inits .append(np.mean(d_inits))
    sd_d_inits.append(np.std(d_inits))    
 
relaxation_t_df["avg_tbreak, ms"] = avg_tbreaks
relaxation_t_df["sd_tbreak"] = sd_tbreaks
relaxation_t_df["avg_d_breaks, mm"] = avg_d_breaks
relaxation_t_df["sd_d_breaks"] = sd_d_breaks
relaxation_t_df["avg_d_prebreaks, mm"] = avg_d_prebreaks
relaxation_t_df["sd_d_prebreaks"] = sd_d_prebreaks
relaxation_t_df["avg_d_inits, mm"] = avg_d_inits
relaxation_t_df["sd_d_inits"] = sd_d_inits
relaxation_t_df

# Export Data as a csv file
# filename = "Data Summary"
# relaxation_t_df.to_csv(fr"{parent_folder}\{filename}.csv", index = False)
relaxation_t_df


,experiment_id,start_ind,end_ind,N,rsq,slope,intercept,"avg relaxation time, ms",sd relaxation time,"avg_tbreak, ms",sd_tbreak,"avg_d_breaks, mm",sd_d_breaks,"avg_d_prebreaks, mm",sd_d_prebreaks,"avg_d_inits, mm",sd_d_inits
0,50kda_mg,12.0,24.0,3.0,0.990081,0.021743,-1.484553,15.360650,0.673591,27.333333,2.867442,0.000,0.000000,0.200667,4.054079e-02,2.393667,0.073086
1,50kda_naoh,11.0,21.0,3.0,0.992381,0.021927,-1.481727,15.306269,1.273009,28.333333,5.436502,0.000,0.000000,0.344000,7.021871e-02,2.422333,0.141893
2,50kda_gel,10.0,22.0,3.0,0.989338,0.022999,-1.512680,14.556118,0.946462,31.333333,6.798693,0.000,0.000000,0.229333,8.835660e-02,2.508333,0.165920
3,50kda_ha,8.0,17.0,4.0,0.987690,0.032966,-1.643693,10.138631,0.523101,28.250000,10.755812,0.000,0.000000,0.236500,9.852538e-02,2.375750,0.211477
4,100kda_mg,11.0,25.0,4.0,0.985881,0.019485,-1.502229,17.495002,2.600855,42.000000,8.573214,0.000,0.000000,0.408500,1.376672e-01,2.644500,0.165145
5,100kda_naoh,9.0,25.0,3.0,0.990599,0.023039,-1.522783,14.474238,0.299135,28.333333,1.699673,0.000,0.000000,0.315333,1.128608e-01,2.422333,0.020270
6,100kda_gel,18.0,40.0,3.0,0.993021,0.012642,-1.413898,26.640849,2.741581,49.666667,8.730534,0.000,0.000000,0.243667,1.072608e-01,2.680333,0.020270
7,100kda_ha,11.0,28.0,4.0,0.991520,0.019367,-1.512581,17.355179,1.635751,34.500000,7.262920,0.000,0.000000,0.258000,1.608913e-01,2.429500,0.064500
8,250kda_mg,23.0,56.0,3.0,0.994904,0.009060,-1.421609,37.049222,3.075984,71.000000,10.677078,0.000,0.000000,0.329667,2.027039e-02,2.737667,0.053630
9,250kda_naoh,11.0,25.0,3.0,0.993270,0.022123,-1.531913,15.117545,0.882782,29.000000,3.741657,0.000,0.000000,0.258000,3.510935e-02,2.379333,0.088357
